# Maintenance Intelligence

This notebook combines the validated priority-classification output, complaint-history signals, affected-user impact, and the existing K-Means clustering artifact to rank maintenance complaints.

**Production alignment:** this notebook does not retrain the classifier, scaler, PCA, or K-Means model. It reuses the saved artifacts already used by the CampusPulse ML API.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd

DATA_PATH = "../data/processed/campuspulse_akgec_processed.csv"
MODEL_DIR = "../ml/models/classification"
CLUSTER_ARTIFACT = "../ml/models/clustering/kmeans_model.json"

df = pd.read_csv(DATA_PATH)
priority_pipeline = joblib.load(f"{MODEL_DIR}/priority_pipeline.pkl")
with open(CLUSTER_ARTIFACT, "r", encoding="utf-8") as f:
    cluster_artifact = json.load(f)

print("Dataset shape:", df.shape)
print("Loaded classification bundle and K-Means artifact.")

Dataset shape: (200000, 41)
Loaded classification bundle and K-Means artifact.


## Priority Prediction

The processed dataset already contains the final 40 classification features, including engineered features. The saved priority pipeline bundle is reused directly.

In [2]:
feature_columns = priority_pipeline["features"]
X = df[feature_columns].copy()

preprocessor = priority_pipeline["preprocessor"]
pca = priority_pipeline["pca"]
model = priority_pipeline["model"]

X_processed = preprocessor.transform(X)
X_pca = pca.transform(X_processed)
df["predicted_priority"] = model.predict(X_pca)

print(df["predicted_priority"].value_counts())

predicted_priority
Medium      93896
Low         58412
High        37593
Critical    10099
Name: count, dtype: int64


## Complaint Intelligence Signals

These are operational signals derived from the complaint record and history. The `similar_issue_history` signal is a frequency/history proxy; it is not semantic text similarity.

In [3]:
priority_map = {"Low": 1, "Medium": 2, "High": 3, "Critical": 4}
df["priority_score"] = df["predicted_priority"].map(priority_map)

df["similar_issue_history"] = (
    df["previous_similar_complaints"].fillna(0)
    + df["complaints_last_7_days"].fillna(0)
    + df["complaints_last_30_days"].fillna(0)
)

df["recurring_issue_signal"] = (
    df["recurrence_count"].fillna(0)
    + df["previous_similar_complaints"].fillna(0)
)

df["affected_user_impact"] = df["affected_users"].fillna(0)

print(df[["predicted_priority", "similar_issue_history", "recurring_issue_signal", "affected_user_impact"]].head())

  predicted_priority  similar_issue_history  recurring_issue_signal  \
0             Medium                    3.0                     2.0   
1             Medium                    5.0                     8.0   
2             Medium                    4.0                     5.0   
3             Medium                    6.0                     7.0   
4               High                    0.0                     0.0   

   affected_user_impact  
0                     9  
1                    17  
2                    49  
3                     5  
4                   145  


In [4]:
def normalize(series):
    min_val = series.min()
    max_val = series.max()
    if max_val == min_val:
        return pd.Series(0.0, index=series.index)
    return (series - min_val) / (max_val - min_val)


df["priority_norm"] = normalize(df["priority_score"])
df["similar_norm"] = normalize(df["similar_issue_history"])
df["recurring_norm"] = normalize(df["recurring_issue_signal"])
df["affected_norm"] = normalize(df["affected_user_impact"])

print("Signals normalized successfully.")

Signals normalized successfully.


## Existing K-Means Pattern Integration

The same K-Means artifact used by the API is loaded from `ml/models/clustering/kmeans_model.json`. No new scaler, PCA, or K-Means model is fitted here.

In [5]:
cluster_features = cluster_artifact["features"]
X_cluster = df[cluster_features].copy().fillna(0)

X_values = X_cluster.to_numpy(dtype=float)
scaler_mean = np.asarray(cluster_artifact["scaler_mean"], dtype=float)
scaler_scale = np.asarray(cluster_artifact["scaler_scale"], dtype=float)
pca_mean = np.asarray(cluster_artifact["pca_mean"], dtype=float)
pca_components = np.asarray(cluster_artifact["pca_components"], dtype=float)
cluster_centers = np.asarray(cluster_artifact["cluster_centers"], dtype=float)

X_scaled = (X_values - scaler_mean) / scaler_scale
X_pca_cluster = (X_scaled - pca_mean) @ pca_components.T

distances = np.linalg.norm(
    X_pca_cluster[:, None, :] - cluster_centers[None, :, :],
    axis=2,
)
df["cluster"] = np.argmin(distances, axis=1)

cluster_names = {
    int(k): v for k, v in cluster_artifact["cluster_names"].items()
}

cluster_descriptions = {
    int(k): v for k, v in cluster_artifact["cluster_descriptions"].items()
}

df["cluster_name"] = df["cluster"].map(cluster_names)
df["cluster_description"] = df["cluster"].map(cluster_descriptions)

print("Cluster distribution:")
print(df["cluster"].value_counts().sort_index())

Cluster distribution:
cluster
0     64732
1    133511
2      1757
Name: count, dtype: int64


In [6]:
cluster_counts = df["cluster"].value_counts()
df["cluster_pattern_frequency"] = df["cluster"].map(cluster_counts)
df["cluster_pattern_norm"] = normalize(df["cluster_pattern_frequency"])

df["similar_pattern_norm"] = (
    0.5 * df["similar_norm"]
    + 0.5 * df["cluster_pattern_norm"]
)

print(df[["cluster", "cluster_name", "cluster_pattern_frequency", "similar_pattern_norm"]].head())

   cluster                        cluster_name  cluster_pattern_frequency  \
0        1       Less recurring and lower-cost                     133511   
1        1       Less recurring and lower-cost                     133511   
2        1       Less recurring and lower-cost                     133511   
3        0  Highly recurring and long-duration                      64732   
4        1       Less recurring and lower-cost                     133511   

   similar_pattern_norm  
0              0.544118  
1              0.573529  
2              0.558824  
3              0.327222  
4              0.500000  


## Final Maintenance Score

The final score is a decision-ranking score, not an ML probability:

- Predicted Priority — 40%
- Similar/Cluster Complaint Pattern — 20%
- Recurring Issue Signal — 20%
- Affected User Impact — 20%

In [7]:
df["maintenance_score"] = (
    0.40 * df["priority_norm"]
    + 0.20 * df["similar_pattern_norm"]
    + 0.20 * df["recurring_norm"]
    + 0.20 * df["affected_norm"]
)

df["maintenance_rank"] = (
    df["maintenance_score"]
    .rank(method="first", ascending=False)
    .astype(int)
)

print("Maintenance score calculated successfully.")

Maintenance score calculated successfully.


In [8]:
top_10 = df.sort_values(
    "maintenance_score",
    ascending=False
).head(10)

display_columns = [
    "predicted_priority",
    "cluster",
    "cluster_name",
    "similar_issue_history",
    "recurring_issue_signal",
    "affected_user_impact",
    "maintenance_score",
    "maintenance_rank",
]

print(top_10[display_columns].to_string(index=False))

predicted_priority  cluster                       cluster_name  similar_issue_history  recurring_issue_signal  affected_user_impact  maintenance_score  maintenance_rank
          Critical        0 Highly recurring and long-duration                   29.0                    25.0                   779           0.736638                 1
          Critical        0 Highly recurring and long-duration                   30.0                    29.0                    14           0.736553                 2
          Critical        2          High-impact and high-cost                   14.0                    14.0                  4939           0.735327                 3
          Critical        2          High-impact and high-cost                   16.0                    12.0                  4972           0.728737                 4
          Critical        2          High-impact and high-cost                   18.0                    11.0                  4981           0.728083     

## Maintenance Intelligence Summary

The maintenance intelligence layer combines the validated supervised priority prediction with complaint-history signals, recurring-issue signals, affected-user impact, and the same K-Means pattern model used by the API.

Higher maintenance scores indicate complaints that should receive faster maintenance attention based on the defined decision weights.

This layer does not train a new ML model; it converts existing ML outputs and operational signals into an actionable maintenance ranking.